# Chapter 15 Companion Notebook: Alternative Data and Deep Learning in Finance

This notebook reproduces every worked example in Chapter 15 from scratch, in the order they appear in the chapter text.

## 1. Vendor subscription cost-benefit calculation

In [ ]:
import numpy as np

signal_freq = 5 / 8          # fraction of quarters the Section 5 signal fires long
releases_per_year = 4
mean_signal_return = 0.88    # % next-day return on signaled quarters (Example 3)
mean_all_return = 0.3375     # % next-day return across all eight quarters (Example 3)
edge_pp = mean_signal_return - mean_all_return
position = 50e6              # $ deployed per signaled event

expected_events_per_year = releases_per_year * signal_freq
expected_annual_profit = position * (edge_pp / 100) * expected_events_per_year

satellite_cost = 250_000
web_card_cost = 150_000
total_cost = satellite_cost + web_card_cost
net_benefit = expected_annual_profit - total_cost

print("edge (percentage points):", edge_pp)
print("expected signaled events/year:", expected_events_per_year)
print("expected annual profit:", expected_annual_profit)
print("total annual data cost:", total_cost)
print("net expected annual benefit:", net_benefit)

## 2. Satellite car-count regression on Cascade's revenue growth

In [2]:
car = np.array([2.1, -1.5, 4.8, 0.6, 6.2, -3.0, 3.5, 1.0])
revenue = np.array([1.5, -0.3, 4.5, 1.0, 4.0, -1.0, 3.8, -0.2])

xbar, ybar = car.mean(), revenue.mean()
Sxy = np.sum((car - xbar) * (revenue - ybar))
Sxx = np.sum((car - xbar) ** 2)
beta = Sxy / Sxx
alpha = ybar - beta * xbar
fitted = alpha + beta * car
r2 = 1 - np.sum((revenue - fitted) ** 2) / np.sum((revenue - ybar) ** 2)

print("xbar, ybar:", xbar, ybar)
print("slope, intercept:", beta, alpha)
print("R^2:", r2)

car_q9 = 5.0
pred_q9 = alpha + beta * car_q9
print("Q9 predicted revenue growth (car=5.0%):", pred_q9)

xbar, ybar: 1.7125 1.6625
slope, intercept: 0.652765135330943 0.5446397057457601
R^2: 0.8699326465048739
Q9 predicted revenue growth (car=5.0%): 3.8084653824004753


## 3. Multi-source earnings-surprise signal

In [3]:
web = np.array([3.0, -0.5, 5.5, 1.5, 4.0, -2.0, 2.5, 2.0])
card = np.array([1.5, -1.0, 3.0, 0.8, 4.5, -1.8, 2.0, 0.5])
consensus = np.array([1.0, 1.0, 2.0, 1.0, 2.0, 0.5, 1.5, 1.0])
surprise = revenue - consensus
next_day_return = np.array([1.1, -0.6, 2.0, 0.3, -0.4, -1.4, 1.5, 0.2])

X = np.column_stack([np.ones(len(car)), car, web, card])
beta_ms, _, _, _ = np.linalg.lstsq(X, surprise, rcond=None)
fitted_ms = X @ beta_ms
r2_ms = 1 - np.sum((surprise - fitted_ms) ** 2) / np.sum((surprise - surprise.mean()) ** 2)

signal = fitted_ms > 0
hit_rate = np.mean(next_day_return[signal] > 0)
mean_signal_ret = next_day_return[signal].mean()
mean_all_ret = next_day_return.mean()

print("beta (intercept, car, web, card):", beta_ms.round(3))
print("R^2:", round(r2_ms, 3))
print("fitted surprise:", fitted_ms.round(2))
print("signal (long if True):", signal)
print("hit rate on signaled quarters:", hit_rate)
print("mean return, signaled vs all:", round(mean_signal_ret, 2), round(mean_all_ret, 2))

car9, web9, card9 = 5.0, 4.5, 3.5
pred_surprise_q9 = np.array([1, car9, web9, card9]) @ beta_ms
print("Q9 predicted surprise:", pred_surprise_q9)

beta (intercept, car, web, card): [-0.291  1.092 -0.113 -0.792]
R^2: 0.837
fitted surprise: [ 0.48 -1.08  1.95 -0.44  2.46 -1.92  1.66  0.18]
signal (long if True): [ True False  True False  True False  True  True]
hit rate on signaled quarters: 0.8
mean return, signaled vs all: 0.88 0.34
Q9 predicted surprise: 1.8882117592166234


## 4. Point-in-time alignment: naive vs. lagged card data

In [4]:
car_b, web_b, card_b, surprise_b, return_b = car[1:], web[1:], card[1:], surprise[1:], next_day_return[1:]
card_lagged_b = card[:-1]

X_naive = np.column_stack([np.ones(7), car_b, web_b, card_b])
beta_naive, _, _, _ = np.linalg.lstsq(X_naive, surprise_b, rcond=None)
fitted_naive = X_naive @ beta_naive
r2_naive = 1 - np.sum((surprise_b - fitted_naive) ** 2) / np.sum((surprise_b - surprise_b.mean()) ** 2)
signal_naive = fitted_naive > 0
hit_naive = np.mean(return_b[signal_naive] > 0)
mean_naive = return_b[signal_naive].mean()

X_real = np.column_stack([np.ones(7), car_b, web_b, card_lagged_b])
beta_real, _, _, _ = np.linalg.lstsq(X_real, surprise_b, rcond=None)
fitted_real = X_real @ beta_real
r2_real = 1 - np.sum((surprise_b - fitted_real) ** 2) / np.sum((surprise_b - surprise_b.mean()) ** 2)
signal_real = fitted_real > 0
hit_real = np.mean(return_b[signal_real] > 0)
mean_real = return_b[signal_real].mean()

print("Naive (contemporaneous card):")
print("  beta:", beta_naive.round(3), "R^2:", round(r2_naive, 3))
print("  signal:", signal_naive, "hit rate:", hit_naive, "mean return:", round(mean_naive, 2))
print()
print("Realistic (one-quarter-lagged card):")
print("  beta:", beta_real.round(3), "R^2:", round(r2_real, 3))
print("  signal:", signal_real, "n signaled:", signal_real.sum(), "hit rate:", round(hit_real, 4), "mean return:", round(mean_real, 2))

Naive (contemporaneous card):
  beta: [-0.29   1.102 -0.119 -0.802] R^2: 0.837
  signal: [False  True False  True False  True  True] hit rate: 0.75 mean return: 0.82

Realistic (one-quarter-lagged card):
  beta: [ 0.251  0.424 -0.1   -0.286] R^2: 0.873
  signal: [False  True False  True False  True False] n signaled: 3 hit rate: 0.6667 mean return: 1.03


## 5. Peer-relative normalization of car-count growth

In [5]:
sector = np.array([1.0, -0.5, 2.0, 0.3, 2.5, -1.0, 1.5, 0.5])
peer_adj = car - sector

xbar_p, ybar_p = peer_adj.mean(), revenue.mean()
Sxy_p = np.sum((peer_adj - xbar_p) * (revenue - ybar_p))
Sxx_p = np.sum((peer_adj - xbar_p) ** 2)
beta_p = Sxy_p / Sxx_p
alpha_p = ybar_p - beta_p * xbar_p
fitted_p = alpha_p + beta_p * peer_adj
r2_p = 1 - np.sum((revenue - fitted_p) ** 2) / np.sum((revenue - ybar_p) ** 2)

print("peer-adjusted car-count growth:", peer_adj)
print("slope, intercept:", round(beta_p, 4), round(alpha_p, 4))
print("R^2:", round(r2_p, 4))

peer-adjusted car-count growth: [ 1.1 -1.   2.8  0.3  3.7 -2.   2.   0.5]
slope, intercept: 1.0639 0.6784
R^2: 0.8666


## 6. A hand-computed 2D convolution over a satellite image patch

In [6]:
image = np.array([
    [0.1, 0.1, 0.8, 0.9],
    [0.1, 0.1, 0.9, 0.8],
    [0.7, 0.8, 0.1, 0.1],
    [0.8, 0.9, 0.1, 0.1],
])
kernel = np.full((2, 2), 0.25)

feature_map = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        patch = image[i:i + 2, j:j + 2]
        feature_map[i, j] = np.sum(patch * kernel)

print("feature map:\n", feature_map.round(3))

feature map:
 [[0.1   0.475 0.85 ]
 [0.425 0.475 0.475]
 [0.8   0.475 0.1  ]]


## 7. A hand-computed RNN forward pass over web-traffic readings

In [7]:
x_seq = [0.5, -0.2, 0.9]
w_x, w_h, b = 0.6, 0.4, 0.1
h = 0.0
hidden_states = []
for x_t in x_seq:
    h = np.tanh(w_x * x_t + w_h * h + b)
    hidden_states.append(h)

w_y, b_y = 1.2, -0.3
y_hat = 1 / (1 + np.exp(-(w_y * hidden_states[-1] + b_y)))

print("hidden states h1, h2, h3:", [round(h, 4) for h in hidden_states])
print("output y_hat:", round(y_hat, 4))

hidden states h1, h2, h3: [np.float64(0.3799), np.float64(0.1312), np.float64(0.5996)]
output y_hat: 0.6034


## 8. Autoencoder reconstruction error for data-quality anomaly flagging

In [8]:
X_ae = np.column_stack([car, web, card])
w_enc = np.array([0.5, 0.3, 0.2])
z = X_ae @ w_enc
x_hat = np.outer(z, w_enc)
recon_error = np.sum((X_ae - x_hat) ** 2, axis=1)
energy = np.sum(X_ae ** 2, axis=1)
rel_error = recon_error / energy

for i in range(8):
    print(f"Q{i+1}: z={z[i]:.4f} recon_error={recon_error[i]:.4f} rel_error={rel_error[i]:.4f}")

print("\nQ4 reconstruction:", x_hat[3].round(3), " error:", round(recon_error[3], 3))
print("quarter with max relative error:", f"Q{np.argmax(rel_error)+1}", round(rel_error.max(), 4))
print("quarter with min relative error:", f"Q{np.argmin(rel_error)+1}", round(rel_error.min(), 4))

Q1: z=2.2500 recon_error=7.4588 rel_error=0.4763
Q2: z=-1.1000 recon_error=1.5398 rel_error=0.4399
Q3: z=4.6500 recon_error=27.2616 rel_error=0.4377
Q4: z=0.9100 recon_error=1.9085 rel_error=0.5872
Q5: z=5.2000 recon_error=30.8852 rel_error=0.4135
Q6: z=-2.4600 recon_error=6.4364 rel_error=0.3963
Q7: z=2.9000 recon_error=8.8758 rel_error=0.3945
Q8: z=1.2000 recon_error=2.9172 rel_error=0.5557

Q4 reconstruction: [0.455 0.273 0.182]  error: 1.908
quarter with max relative error: Q4 0.5872
quarter with min relative error: Q7 0.3945


## 8b. Vision transformer patch self-attention (Section 15.8)

Splitting the same 4x4 satellite image from the CNN example into four non-overlapping 2x2 patches (a vision transformer's "tokens"), then applying scaled dot-product self-attention across patches, exactly the mechanism Chapter 14 hand-computed for text tokens. Querying from patch 2 (the top-right car cluster) should attend most to patch 3 (the diagonally distant bottom-left car cluster), not to the spatially nearer but visually dissimilar empty patches 1 and 4, directly illustrating how self-attention lets a model weigh a distant, content-relevant patch directly rather than only through many stacked local convolutions.

In [9]:
P1, P2, P3, P4 = image[0:2, 0:2], image[0:2, 2:4], image[2:4, 0:2], image[2:4, 2:4]
patch_means = [P1.mean(), P2.mean(), P3.mean(), P4.mean()]
print("patch means (P1 top-left, P2 top-right, P3 bottom-left, P4 bottom-right):",
      [round(m, 3) for m in patch_means])

# Illustrative patch embeddings: keys/query carry only content (brightness), scaled
# as a learned Q/K projection would; values carry brightness plus a patch-index tag
scale = 4.0
K_vit = np.array([[m * scale, 0.0] for m in patch_means])
V_vit = np.array([[m, i + 1] for i, m in enumerate(patch_means)])
Q_patch2 = np.array([patch_means[1] * scale, 0.0])

d_k_vit = 2
scores_vit = (K_vit @ Q_patch2) / np.sqrt(d_k_vit)
weights_vit = np.exp(scores_vit) / np.exp(scores_vit).sum()
output_vit = weights_vit @ V_vit

print("scores:", scores_vit.round(4))
print("softmax attention weights (P1,P2,P3,P4):", weights_vit.round(4))
print("attention output for patch 2 (querying):", output_vit.round(4))

patch means (P1 top-left, P2 top-right, P3 bottom-left, P4 bottom-right): [np.float64(0.1), np.float64(0.85), np.float64(0.8), np.float64(0.1)]
scores: [0.9617 8.1742 7.6933 0.9617]
softmax attention weights (P1,P2,P3,P4): [5.000e-04 6.174e-01 3.817e-01 5.000e-04]
attention output for patch 2 (querying): [0.8302 2.3822]


## 9. Transfer learning: one fine-tuning gradient step on a frozen backbone

In [10]:
f = np.array([0.7, 0.3])   # frozen backbone features
y_true = 1.0
w = np.array([0.4, 0.4])
b_tl = -0.1

z = w @ f + b_tl
y_hat = 1 / (1 + np.exp(-z))

eta = 0.5
grad_w = (y_hat - y_true) * f
grad_b = (y_hat - y_true)
w_new = w - eta * grad_w
b_new = b_tl - eta * grad_b

z_new = w_new @ f + b_new
y_hat_new = 1 / (1 + np.exp(-z_new))

print("initial prediction:", round(y_hat, 4))
print("gradients (w, b):", grad_w.round(3), round(grad_b, 3))
print("updated weights:", w_new.round(3), "updated bias:", round(b_new, 3))
print("updated prediction:", round(y_hat_new, 4))

initial prediction: 0.5744
gradients (w, b): [-0.298 -0.128] -0.426
updated weights: [0.549 0.464] updated bias: 0.113
updated prediction: 0.6539


## 10. Graph neural network mean-aggregation over a shell-company chain

In [11]:
x = {"A": 0.9, "B": 0.2, "C": 0.1, "D": 0.05}
neighbors = {"A": ["B"], "B": ["A", "C"], "C": ["B", "D"], "D": ["C"]}
w1, w2, b_gnn = 0.6, 0.8, -0.05

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

h = {}
for node in x:
    neighbor_mean = np.mean([x[n] for n in neighbors[node]])
    z_node = w1 * x[node] + w2 * neighbor_mean + b_gnn
    h[node] = sigmoid(z_node)

for node in ["A", "B", "C", "D"]:
    print(f"h_{node} = {h[node]:.4f} (raw feature x_{node} = {x[node]})")

h_A = 0.6570 (raw feature x_A = 0.9)
h_B = 0.6154 (raw feature x_B = 0.2)
h_C = 0.5275 (raw feature x_C = 0.1)
h_D = 0.5150 (raw feature x_D = 0.05)


## 11. Multi-modal fusion via cross-attention

In [12]:
Q = np.array([0.6, 0.8])
K_img, V_img = np.array([0.9, 0.1]), np.array([0.7, 0.2])
K_txt, V_txt = np.array([0.2, 0.9]), np.array([-0.3, 0.6])
d_k = 2

score_img = (Q @ K_img) / np.sqrt(d_k)
score_txt = (Q @ K_txt) / np.sqrt(d_k)
e_img, e_txt = np.exp(score_img), np.exp(score_txt)
w_img, w_txt = e_img / (e_img + e_txt), e_txt / (e_img + e_txt)
fused = w_img * V_img + w_txt * V_txt

print("scores (image, text):", round(score_img, 4), round(score_txt, 4))
print("attention weights (image, text):", round(w_img, 4), round(w_txt, 4))
print("fused representation:", fused.round(3))

scores (image, text): 0.4384 0.594
attention weights (image, text): 0.4612 0.5388
fused representation: [0.161 0.416]


## 12. Self-supervised contrastive pretraining probability

In [13]:
a = np.array([0.6, 0.8])
p = np.array([0.55, 0.83])
n = np.array([0.1, 0.9])

def cosine(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

cos_ap, cos_an = cosine(a, p), cosine(a, n)

tau = 0.1
e_p, e_n = np.exp(cos_ap / tau), np.exp(cos_an / tau)
prob_positive = e_p / (e_p + e_n)

print("cos(a, p):", round(cos_ap, 4))
print("cos(a, n):", round(cos_an, 4))
print("Pr(positive) at tau=0.1:", round(prob_positive, 4))

cos(a, p): 0.9983
cos(a, n): 0.8614
Pr(positive) at tau=0.1: 0.7973
